In [2]:
"""
Vietnam — Ăsta nu-i pantoful meu! (Open-Set Sneaker Recognition -> main.pt <= 10 MB)
Run in Google Colab (GPU T4 recommended).
1. Downloads the 735 training images + train-output.csv fast via Drive REST + requests (~3s, no warnings).
2. Trains a MobileNetV3-Small classifier (<= 7 MB, well under 10 MB limit) on 256x256 sneakers
   for the 3 known classes: adidas, converse, nike.
3. Calibrates an Open-Set Recognition threshold using Energy + Max-Logit on a stratified
   validation split (targeting ~92% known TPR).
4. Exports a self-contained TorchScript model (`main.pt`, <= 10 MB) with ImageNet normalization
   and [0,1] / [0,255] auto-scaling baked into `forward()`, outputting 4-class logits
   [adidas, converse, nike, unknown] where index 3 wins whenever energy < threshold.
"""
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

FOLDER_ID = "12INJs3C0w8eWLOWy43rP74HhHBLghhbF"
CLASSES = ["adidas", "converse", "nike"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


def _find_in_tree(base: Path, target: str = "train-output.csv") -> Path | None:
    if not base.exists():
        return None
    for dirpath, _, filenames in os.walk(base, followlinks=True):
        if target in filenames:
            return Path(dirpath)
    return None


def _download_via_colab_drive_api(dest: Path) -> Path:
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    auth.authenticate_user()
    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    creds.refresh(google.auth.transport.requests.Request())
    headers = {"Authorization": f"Bearer {creds.token}"}
    session = requests.Session()
    session.headers.update(headers)

    def list_children(parent_id: str) -> list[dict]:
        items, page_token = [], None
        while True:
            params = {
                "q": f"'{parent_id}' in parents and trashed=false",
                "fields": "nextPageToken, files(id, name, mimeType)",
                "pageSize": 1000,
            }
            if page_token:
                params["pageToken"] = page_token
            r = session.get("https://www.googleapis.com/drive/v3/files", params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            items.extend(data.get("files", []))
            page_token = data.get("nextPageToken")
            if not page_token:
                break
        return items

    def download_file(file_id: str, out_path: Path) -> None:
        if out_path.exists() and out_path.stat().st_size > 0:
            return
        r = session.get(f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media", timeout=30)
        r.raise_for_status()
        out_path.write_bytes(r.content)

    dest.mkdir(parents=True, exist_ok=True)
    tasks: list[tuple[str, Path]] = []
    for item in list_children(FOLDER_ID):
        if item["mimeType"] == "application/vnd.google-apps.folder":
            sub_dir = dest / item["name"]
            sub_dir.mkdir(parents=True, exist_ok=True)
            for child in list_children(item["id"]):
                tasks.append((child["id"], sub_dir / child["name"]))
        else:
            tasks.append((item["id"], dest / item["name"]))

    print(f"Downloading {len(tasks)} files into {dest}...")
    with ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(lambda args: download_file(*args), tasks))
    print("Download complete.")
    return dest


def ensure_data() -> Path:
    for candidate in [
        Path("/content/task_vietnam"),
        Path("/content"),
        Path(f"/content/drive/.shortcut-targets-by-id/{FOLDER_ID}"),
        Path("/content/drive/MyDrive"),
    ]:
        found = _find_in_tree(candidate)
        if found is not None:
            return found
    return _download_via_colab_drive_api(Path("/content/task_vietnam"))


class SneakerDataset(Dataset):
    def __init__(self, img_dir: Path, df: pd.DataFrame, tfm):
        self.img_dir = img_dir
        self.df = df.reset_index(drop=True)
        self.tfm = tfm

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row["filename"]).convert("RGB")
        return self.tfm(img), CLASS_TO_IDX[row["label"]]


class OpenSetSneakerModel(nn.Module):
    """Self-contained TorchScript-compatible Open-Set Sneaker Classifier."""

    def __init__(self, backbone: nn.Module, energy_threshold: float):
        super().__init__()
        self.backbone = backbone
        self.classes = ["adidas", "converse", "nike", "unknown"]
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.energy_threshold = float(energy_threshold)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.max() > 1.5:
            x = x / 255.0
        x = (x - self.mean) / self.std
        logits = self.backbone(x)  # (B, 3)
        energy = torch.logsumexp(logits, dim=1, keepdim=True)
        max_logit = logits.max(dim=1, keepdim=True).values
        unknown_logit = max_logit + (self.energy_threshold - energy)
        return torch.cat([logits, unknown_logit], dim=1)  # (B, 4)


def main() -> None:
    root = ensure_data()
    print("Dataset root:", root)
    df = pd.read_csv(root / "train-output.csv")
    img_dir = root / "train-input"

    rng = np.random.default_rng(42)
    trn_idx, val_idx = [], []
    for _, grp in df.groupby("label"):
        ids = grp.index.to_numpy().copy()
        rng.shuffle(ids)
        n_val = max(1, int(0.15 * len(ids)))
        val_idx.extend(ids[:n_val])
        trn_idx.extend(ids[n_val:])

    trn_tfm = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
    ])
    val_tfm = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])

    trn_loader = DataLoader(SneakerDataset(img_dir, df.iloc[trn_idx], trn_tfm), batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(SneakerDataset(img_dir, df.iloc[val_idx], val_tfm), batch_size=32, shuffle=False, num_workers=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    backbone.classifier[-1] = nn.Linear(backbone.classifier[-1].in_features, len(CLASSES))
    backbone = backbone.to(device)

    mean_t = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std_t = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    optimizer = torch.optim.AdamW(backbone.parameters(), lr=5e-4, weight_decay=1e-4)
    epochs = 25
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    best_state = None
    for epoch in range(1, epochs + 1):
        backbone.train()
        for imgs, labels in trn_loader:
            imgs, labels = (imgs.to(device) - mean_t) / std_t, labels.to(device)
            loss = F.cross_entropy(backbone(imgs), labels, label_smoothing=0.05)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

        backbone.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = (imgs.to(device) - mean_t) / std_t, labels.to(device)
                preds = backbone(imgs).argmax(dim=1)
                correct += int((preds == labels).sum())
                total += len(labels)
        acc = correct / total
        if acc >= best_acc:
            best_acc = acc
            best_state = {k: v.cpu().clone() for k, v in backbone.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d}/{epochs} | Val Known Acc: {acc:.4f} (best: {best_acc:.4f})")

    backbone.load_state_dict(best_state)
    backbone = backbone.to(device).eval()

    val_energies = []
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = (imgs.to(device) - mean_t) / std_t
            logits = backbone(imgs)
            val_energies.extend(torch.logsumexp(logits, dim=1).cpu().numpy().tolist())
    energy_thresh = float(np.percentile(val_energies, 8))
    print(f"Calibrated energy threshold (8th pct): {energy_thresh:.4f}")

    wrapped = OpenSetSneakerModel(backbone.cpu(), energy_thresh).eval()
    scripted = torch.jit.script(wrapped)
    out_pt = Path("/content/main.pt")
    scripted.save(str(out_pt))
    size_mb = out_pt.stat().st_size / (1024 * 1024)
    print(f"Saved {out_pt} ({size_mb:.2f} MB, limit 10.00 MB)")

    loaded = torch.jit.load(str(out_pt))
    dummy = torch.rand(2, 3, 256, 256)
    out = loaded(dummy)
    print("Reloaded TorchScript output shape:", tuple(out.shape))

    try:
        from google.colab import files
        files.download(str(out_pt))
    except ImportError:
        pass


if __name__ == "__main__":
    main()

Download complete.
Dataset root: /content/task_vietnam
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 89.0MB/s]


Epoch 01/25 | Val Known Acc: 0.5648 (best: 0.5648)
Epoch 05/25 | Val Known Acc: 0.8148 (best: 0.8148)
Epoch 10/25 | Val Known Acc: 0.8148 (best: 0.8519)
Epoch 15/25 | Val Known Acc: 0.8426 (best: 0.8611)
Epoch 20/25 | Val Known Acc: 0.9074 (best: 0.9074)
Epoch 25/25 | Val Known Acc: 0.9074 (best: 0.9074)
Calibrated energy threshold (8th pct): 1.3811


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Saved /content/main.pt (6.08 MB, limit 10.00 MB)


/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Reloaded TorchScript output shape: (2, 4)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>